In [11]:
def answer_one():
    import pandas as pd
    import numpy as np

    pd.set_option('future.no_silent_downcasting', True)

    energy = pd.read_excel('Energy Indicators.xls', skiprows=17, skipfooter=38)
    energy = energy.drop(energy.columns[[0, 1]], axis=1)
    energy.columns = ['Country', 'Energy Supply', 'Energy Supply per Capita', '% Renewable']
    
    energy = energy.replace('...', np.nan)
    energy['Energy Supply'] = pd.to_numeric(energy['Energy Supply']) * 1000000
    
    energy['Country'] = energy['Country'].str.replace(r" \(.*\)", "", regex=True)
    energy['Country'] = energy['Country'].str.replace(r"\d+", "", regex=True)
    energy['Country'] = energy['Country'].str.strip()

    re_name = {
        "Republic of Korea": "South Korea",
        "United States of America": "United States",
        "United Kingdom of Great Britain and Northern Ireland": "United Kingdom",
        "China, Hong Kong Special Administrative Region": "Hong Kong"
    }
    energy['Country'] = energy['Country'].replace(re_name)

    gdp = pd.read_excel('world_bank.xls', sheet_name='Data', skiprows=3)
    gdp['Country Name'] = gdp['Country Name'].replace({
        "Korea, Rep.": "South Korea", 
        "Iran, Islamic Rep.": "Iran",
        "Hong Kong SAR, China": "Hong Kong"
    })

    scimago = pd.read_excel('scimagojr.xlsx')

    df = pd.merge(scimago, energy, how='inner', on='Country')
    df = pd.merge(df, gdp, how='inner', left_on='Country', right_on='Country Name')

    years_to_keep = [y for y in [2006.0, 2007.0, 2008.0, 2009.0, 2010.0, 2011.0, 2012.0, 2013.0, 2014.0, 2015.0] if y in df.columns]
    
    if not years_to_keep:
        years_to_keep = [str(y) for y in range(2006, 2016) if str(y) in df.columns]
    if not years_to_keep:
        years_to_keep = [y for y in range(2006, 2016) if y in df.columns]

    cols = ['Rank', 'Documents', 'Citable documents', 'Citations', 'Self-citations', 
            'Citations per document', 'H index', 'Energy Supply', 'Energy Supply per Capita', '% Renewable'] + years_to_keep
    
    df = df.set_index('Country')
    df = df[cols].sort_values('Rank').head(15)
    
    year_rename = {old: str(int(old)) if isinstance(old, (float, int)) else old for old in years_to_keep}
    df = df.rename(columns=year_rename)
    
    return df

answer_one()

,Rank,Documents,Citable documents,Citations,Self-citations,Citations per document,H index,Energy Supply,Energy Supply per Capita,% Renewable,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015
Country,,,,,,,,,,,,,,,,,,,,
China,1,273437,272374,2336764,1615239,8.55,245,1.271910e+11,93,19.754910,2.752119e+12,3.550328e+12,4.594337e+12,5.101691e+12,6.087192e+12,7.551545e+12,8.532186e+12,9.570471e+12,1.047562e+13,1.106157e+13
United States,2,175891,172431,2230544,724472,12.68,363,9.083800e+10,286,11.570980,1.381559e+13,1.447423e+13,1.476986e+13,1.447806e+13,1.504896e+13,1.559973e+13,1.625397e+13,1.684319e+13,1.755068e+13,1.820602e+13
India,3,55082,53775,463165,162944,8.41,181,3.319500e+10,26,14.969080,9.402599e+11,1.216736e+12,1.198895e+12,1.341888e+12,1.675616e+12,1.823052e+12,1.827638e+12,1.856721e+12,2.039126e+12,2.103588e+12
Japan,4,50523,50065,488062,119930,9.66,193,1.898400e+10,149,10.232820,4.601663e+12,4.579750e+12,5.106679e+12,5.289494e+12,5.759072e+12,6.233147e+12,6.272363e+12,5.212328e+12,4.896994e+12,4.444931e+12
United Kingdom,5,43389,42284,615670,111290,14.19,226,7.920000e+09,124,10.600470,2.709978e+12,3.092996e+12,2.931684e+12,2.417566e+12,2.491397e+12,2.666403e+12,2.706341e+12,2.786315e+12,3.065223e+12,2.934858e+12
Germany,6,38739,38013,433148,95145,11.18,196,1.326100e+10,165,17.901530,2.994704e+12,3.425578e+12,3.745264e+12,3.411261e+12,3.399668e+12,3.749315e+12,3.527143e+12,3.733805e+12,3.889093e+12,3.357586e+12
Russian Federation,7,36735,36560,115938,54993,3.16,90,3.070900e+10,214,17.288680,9.899321e+11,1.299703e+12,1.660848e+12,1.222646e+12,1.524917e+12,2.045923e+12,2.208294e+12,2.292470e+12,2.059242e+12,1.363482e+12
Canada,8,33472,32863,568080,100953,16.97,227,1.043100e+10,296,61.945430,1.319265e+12,1.468820e+12,1.552990e+12,1.374625e+12,1.617343e+12,1.793327e+12,1.828366e+12,1.846597e+12,1.805750e+12,1.556509e+12
Italy,9,27983,26940,352993,87828,12.61,166,6.530000e+09,109,33.667230,1.949552e+12,2.213102e+12,2.408655e+12,2.199929e+12,2.136100e+12,2.294994e+12,2.086958e+12,2.141924e+12,2.162010e+12,1.836638e+12


In [12]:
def answer_two():
    Top15 = answer_one()
    years = [str(y) for y in range(2006, 2016)]
    avgGDP = Top15[years].mean(axis=1).sort_values(ascending=False)
    avgGDP.name = 'avgGDP'
    return avgGDP

answer_two()

Country
United States         1.570403e+13
China                 6.927707e+12
Japan                 5.239642e+12
Germany               3.523342e+12
United Kingdom        2.780276e+12
France                2.691337e+12
Italy                 2.142986e+12
Brazil                1.988889e+12
Russian Federation    1.666746e+12
Canada                1.616359e+12
India                 1.602352e+12
Spain                 1.400886e+12
South Korea           1.221372e+12
Australia             1.207513e+12
Iran                  4.563261e+11
Name: avgGDP, dtype: float64

In [13]:
def answer_three():
    Top15 = answer_one()
    avgGDP = answer_two()
    target_country = avgGDP.index[5]
    result = Top15.loc[target_country, '2015'] - Top15.loc[target_country, '2006']
    return result
answer_three()

np.float64(118652421857.7959)

In [14]:
def answer_four():
    Top15 = answer_one()
    Top15['Ratio'] = Top15['Self-citations'] / Top15['Citations']
    max_ratio = Top15['Ratio'].max()
    max_country = Top15['Ratio'].idxmax()
    return (max_country, max_ratio)
answer_four()

('China', 0.6912289816173135)

In [15]:
def answer_five():
    Top15 = answer_one()
    Top15['PopEst'] = Top15['Energy Supply'] / Top15['Energy Supply per Capita']
    result = Top15['PopEst'].sort_values(ascending=False).index[2]
    return result
answer_five()

'United States'

In [16]:
def answer_six():
    Top15 = answer_one()
    Top15['PopEst'] = Top15['Energy Supply'] / Top15['Energy Supply per Capita']
    Top15['Citable docs per Capita'] = Top15['Citable documents'] / Top15['PopEst']
    result = Top15['Citable docs per Capita'].corr(Top15['Energy Supply per Capita'])
    return result
answer_six()

np.float64(0.7434709127726777)

In [18]:
def answer_seven():
    Top15 = answer_one()
    ContinentDict  = {'China':'Asia', 
                      'United States':'North America', 
                      'Japan':'Asia', 
                      'United Kingdom':'Europe', 
                      'Russian Federation':'Europe', 
                      'Canada':'North America', 
                      'Germany':'Europe', 
                      'India':'Asia',
                      'France':'Europe', 
                      'South Korea':'Asia', 
                      'Italy':'Europe', 
                      'Spain':'Europe', 
                      'Iran':'Asia',
                      'Australia':'Australia', 
                      'Brazil':'South America'}
    
    Top15['PopEst'] = (Top15['Energy Supply'] / Top15['Energy Supply per Capita']).astype(float)
    Top15['Continent'] = [ContinentDict[country] for country in Top15.index]
    
    result = Top15.groupby('Continent')['PopEst'].agg(['size', 'sum', 'mean', 'std'])
    
    return result

answer_seven()

,size,sum,mean,std
Continent,,,,
Asia,5,2.898666e+09,5.797333e+08,6.790979e+08
Australia,1,2.331602e+07,2.331602e+07,NaN
Europe,6,4.579297e+08,7.632161e+07,3.464767e+07
North America,2,3.528552e+08,1.764276e+08,1.996696e+08
South America,1,2.059153e+08,2.059153e+08,NaN
